In [1]:
import os
print(os.getcwd())

C:\Users\shahj\QuantTrack


In [2]:
import yfinance as yf
from datetime import date
import sqlite3
import pandas as pd

def fetch_stock_data(ticker, start_date="2015-01-01"):
    """Fetches historical adjusted price data for a given stock ticker.
    Returns None if the ticker is invalid or no data is found."""
    data = yf.download(ticker, start=start_date, end=date.today(), auto_adjust=True)
    
    if data.empty:
        print(f"No data found for ticker '{ticker}'. Check the symbol and try again.")
        return None
    
    data.columns = data.columns.get_level_values(0)
    return data

In [3]:
reliance_data = fetch_stock_data("RELIANCE.NS")
reliance_data.tail()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Date,,,,,
2026-08-18,1322.000000,1328.599976,1311.199951,1314.000000,10180567
2026-08-19,1311.000000,1321.599976,1303.500000,1320.300049,7489365
2026-08-20,1313.199951,1316.500000,1307.000000,1315.500000,5546227
2026-08-21,1316.000000,1316.000000,1308.000000,1314.000000,5434871
2026-08-24,1309.800049,1320.000000,1302.400024,1316.599976,7277167


In [4]:
conn = sqlite3.connect("quanttrack.db")
print("Database connected")

Database connected


In [5]:
reliance_data.to_sql("stock_data", conn, if_exists="replace", index=True)
print("Data saved to database")

Data saved to database


In [6]:
check = pd.read_sql("SELECT * FROM stock_data LIMIT 5", conn)
check

,Date,Close,High,Low,Open,Volume
0,2015-01-01 00:00:00,189.125381,189.998697,188.220116,188.784586,2963643
1,2015-01-02 00:00:00,188.624817,190.861356,188.358571,189.167973,7331366
2,2015-01-05 00:00:00,186.558670,189.764373,186.185911,188.507649,10103941
3,2015-01-06 00:00:00,178.091812,185.951626,177.218496,185.312621,18627980
4,2015-01-07 00:00:00,181.968460,182.926975,178.283514,178.304813,20720312


In [7]:
def moving_average_crossover(data, short_window=20, long_window=50):
    """Calculates short and long moving averages and generates buy/sell signals."""
    df = data.copy()
    
    df["SMA_short"] = df["Close"].rolling(window=short_window).mean()
    df["SMA_long"] = df["Close"].rolling(window=long_window).mean()
    
    # 1 = short average is above long average (bullish), 0 = below (bearish)
    df["Signal"] = 0
    df.loc[df["SMA_short"] > df["SMA_long"], "Signal"] = 1
    
    # Position changes: +1 means a fresh buy signal today, -1 means a fresh sell signal today
    df["Position"] = df["Signal"].diff()
    
    return df

In [8]:
result = moving_average_crossover(reliance_data)
result[["Close", "SMA_short", "SMA_long"]].tail(10)

Price,Close,SMA_short,SMA_long
Date,,,
2026-08-11,1323.900024,1300.504999,1300.773127
2026-08-12,1329.000000,1302.179999,1301.210002
2026-08-13,1317.000000,1303.200000,1301.596003
2026-08-14,1310.000000,1302.340002,1301.976003
2026-08-17,1316.000000,1301.985004,1303.030002
2026-08-18,1322.000000,1302.900006,1304.086003
2026-08-19,1311.000000,1304.020007,1305.130002
2026-08-20,1313.199951,1306.070007,1306.134001
2026-08-21,1316.000000,1307.970007,1306.594001


In [9]:
result = moving_average_crossover(reliance_data)
crossovers = result[result["Position"] != 0]
crossovers[["Close", "SMA_short", "SMA_long", "Position"]].tail(10)

Price,Close,SMA_short,SMA_long,Position
Date,,,,
2025-08-01,1381.767090,1441.397131,1441.487393,-1.0
2025-10-10,1375.340942,1378.864709,1378.543196,1.0
2025-10-13,1368.671875,1377.869312,1377.956101,-1.0
2025-10-23,1441.734009,1384.065668,1382.880029,1.0
2026-01-15,1452.086182,1522.266650,1524.829812,-1.0
2026-05-08,1428.594727,1380.880389,1378.146023,1.0
2026-06-01,1313.924927,1362.201715,1363.320564,-1.0
2026-08-12,1329.000000,1302.179999,1301.210002,1.0
2026-08-17,1316.000000,1301.985004,1303.030002,-1.0


In [10]:
def momentum_strategy(data, lookback=20, threshold=0.05):
    """Generates buy signals when price has risen more than threshold% over the lookback period."""
    df = data.copy()
    
    # % change compared to price 'lookback' days ago
    df["Momentum"] = df["Close"].pct_change(periods=lookback)
    
    df["Signal"] = 0
    df.loc[df["Momentum"] > threshold, "Signal"] = 1
    
    df["Position"] = df["Signal"].diff()
    
    return df

In [11]:
momentum_result = momentum_strategy(reliance_data)
momentum_crossovers = momentum_result[momentum_result["Position"] != 0]
momentum_crossovers[["Close", "Momentum", "Position"]].tail(10)

Price,Close,Momentum,Position
Date,,,
2025-11-20,1541.970581,0.057262,1.0
2025-11-25,1532.613770,0.037464,-1.0
2025-11-26,1562.674805,0.055821,1.0
2025-11-27,1556.204712,0.039357,-1.0
2025-11-28,1560.285889,0.053074,1.0
2025-12-02,1539.183472,0.041490,-1.0
2026-04-29,1418.839844,0.057340,1.0
2026-05-01,1424.215088,0.044990,-1.0
2026-05-04,1456.366333,0.083377,1.0


In [12]:
def mean_reversion_strategy(data, window=20, std_multiplier=1):
    """Generates buy signals when price drops more than std_multiplier standard deviations below the rolling mean."""
    df = data.copy()
    
    df["Rolling_Mean"] = df["Close"].rolling(window=window).mean()
    df["Rolling_Std"] = df["Close"].rolling(window=window).std()
    df["Lower_Band"] = df["Rolling_Mean"] - (std_multiplier * df["Rolling_Std"])
    
    df["Signal"] = 0
    df.loc[df["Close"] < df["Lower_Band"], "Signal"] = 1
    
    df["Position"] = df["Signal"].diff()
    
    return df

In [13]:
reversion_result = mean_reversion_strategy(reliance_data)
reversion_signals = reversion_result[reversion_result["Position"] != 0]
reversion_signals[["Close", "Rolling_Mean", "Lower_Band", "Position"]].tail(10)

Price,Close,Rolling_Mean,Lower_Band,Position
Date,,,,
2026-04-13,1309.047485,1362.186798,1325.215935,1.0
2026-04-15,1337.914062,1359.792871,1322.880911,-1.0
2026-05-15,1330.249512,1385.434326,1341.786227,1.0
2026-05-20,1353.442139,1382.428210,1335.567467,-1.0
2026-05-29,1315.119385,1369.323785,1324.097583,1.0
2026-06-12,1293.000000,1313.577362,1280.570065,-1.0
2026-07-08,1275.900024,1308.195001,1290.767518,1.0
2026-07-10,1307.800049,1309.775006,1294.812508,-1.0
2026-07-22,1288.599976,1303.220001,1289.341266,1.0


In [14]:
def calculate_equity_curve(data):
    """Takes a dataframe with a 'Signal' column and computes daily strategy returns and cumulative equity."""
    df = data.copy()
    
    # Daily % change in the actual stock price
    df["Daily_Return"] = df["Close"].pct_change()
    
    # Shift signal by 1 day: we can only act on yesterday's signal, not today's
    df["Strategy_Return"] = df["Signal"].shift(1) * df["Daily_Return"]
    
    # Cumulative growth of ₹1 invested, following the strategy
    df["Equity_Curve"] = (1 + df["Strategy_Return"]).cumprod()
    
    return df

In [15]:
equity_result = calculate_equity_curve(result)  # 'result' is your MA crossover output from earlier
equity_result[["Close", "Signal", "Strategy_Return", "Equity_Curve"]].tail(10)

Price,Close,Signal,Strategy_Return,Equity_Curve
Date,,,,
2026-08-11,1323.900024,0,-0.000000,2.537131
2026-08-12,1329.000000,1,0.000000,2.537131
2026-08-13,1317.000000,1,-0.009029,2.514222
2026-08-14,1310.000000,1,-0.005315,2.500859
2026-08-17,1316.000000,0,0.004580,2.512313
2026-08-18,1322.000000,0,0.000000,2.512313
2026-08-19,1311.000000,0,-0.000000,2.512313
2026-08-20,1313.199951,0,0.000000,2.512313
2026-08-21,1316.000000,1,0.000000,2.512313


In [16]:
import numpy as np

def calculate_metrics(equity_df):
    """Computes the five key performance metrics from an equity curve."""
    returns = equity_df["Strategy_Return"].dropna()
    
    # Total Return: final equity value minus the starting ₹1
    total_return = equity_df["Equity_Curve"].iloc[-1] - 1
    
    # Sharpe Ratio: average return / volatility of returns, annualized (assuming 252 trading days/year)
    sharpe_ratio = (returns.mean() / returns.std()) * np.sqrt(252)
    
    # Max Drawdown: worst peak-to-trough decline in the equity curve
    running_max = equity_df["Equity_Curve"].cummax()
    drawdown = (equity_df["Equity_Curve"] - running_max) / running_max
    max_drawdown = drawdown.min()
    
    # Volatility: annualized standard deviation of daily returns
    volatility = returns.std() * np.sqrt(252)
    
    # Win Rate: % of active trading days that were profitable
    active_returns = returns[returns != 0]
    win_rate = (active_returns > 0).sum() / len(active_returns)
    
    return {
        "Total Return (%)": round(total_return * 100, 2),
        "Sharpe Ratio": round(sharpe_ratio, 2),
        "Max Drawdown (%)": round(max_drawdown * 100, 2),
        "Volatility (%)": round(volatility * 100, 2),
        "Win Rate (%)": round(win_rate * 100, 2)
    }

In [17]:
metrics = calculate_metrics(equity_result)
metrics

{'Total Return (%)': np.float64(150.05),
 'Sharpe Ratio': np.float64(0.53),
 'Max Drawdown (%)': -41.78,
 'Volatility (%)': np.float64(18.43),
 'Win Rate (%)': np.float64(50.84)}

In [18]:
!pip install ta

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29496 sha256=e0bf0b7b3e2c4737d9b430856aab3481a59338f59aa5776ceaa283d724901624
  Stored in directory: c:\users\shahj\appdata\local\pip\cache\wheels\e3\3a\ee\4955a26c90a4b7deb6d725dc8ec7b8604a7aef44e43a2e8af7
Successfully built ta


In [19]:
from ta.trend import ADXIndicator

def add_adx(data, period=14):
    """Adds ADX (trend strength) to the dataframe."""
    df = data.copy()
    
    adx_indicator = ADXIndicator(high=df["High"], low=df["Low"], close=df["Close"], window=period)
    df["ADX"] = adx_indicator.adx()
    
    return df

In [20]:
adx_result = add_adx(reliance_data)
adx_result[["Close", "ADX"]].tail(15)

Price,Close,ADX
Date,,
2026-08-04,1290.900024,14.492163
2026-08-05,1280.000000,14.850927
2026-08-06,1325.000000,13.929856
2026-08-07,1334.800049,13.279589
2026-08-10,1327.300049,12.675770
2026-08-11,1323.900024,11.778463
2026-08-12,1329.000000,11.157139
2026-08-13,1317.000000,10.675919
2026-08-14,1310.000000,10.507007


In [21]:
def classify_regime(adx_value, threshold=25):
    """Classifies market regime based on ADX value."""
    if adx_value >= threshold:
        return "Trending"
    else:
        return "Sideways"

def recommend_strategy(regime):
    """Recommends a strategy type based on market regime."""
    if regime == "Trending":
        return "Moving Average Crossover or Momentum (trend-following strategies suit trending markets)"
    else:
        return "Mean Reversion (suits sideways/ranging markets)"

In [22]:
latest_adx = adx_result["ADX"].iloc[-1]
current_regime = classify_regime(latest_adx)
recommendation = recommend_strategy(current_regime)

print(f"Latest ADX: {latest_adx:.2f}")
print(f"Market Regime: {current_regime}")
print(f"Recommended Approach: {recommendation}")

Latest ADX: 10.31
Market Regime: Sideways
Recommended Approach: Mean Reversion (suits sideways/ranging markets)


In [23]:
def run_portfolio_backtest(tickers, strategy_func, start_date="2015-01-01"):
    """Runs a given strategy on multiple tickers, returns a dict of equity results per stock."""
    results = {}
    
    for ticker in tickers:
        data = fetch_stock_data(ticker, start_date)
        if data is None:
            print(f"Skipping {ticker} — no data")
            continue
        
        strategy_result = strategy_func(data)
        equity_df = calculate_equity_curve(strategy_result)
        results[ticker] = equity_df
    
    return results

In [24]:
tickers = ["RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS", "ITC.NS"]
portfolio_results = run_portfolio_backtest(tickers, moving_average_crossover)

for ticker, df in portfolio_results.items():
    final_equity = df["Equity_Curve"].iloc[-1]
    print(f"{ticker}: Final Equity = {final_equity:.2f}")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

RELIANCE.NS: Final Equity = 2.50
TCS.NS: Final Equity = 0.81
INFY.NS: Final Equity = 2.08
HDFCBANK.NS: Final Equity = 1.90
ITC.NS: Final Equity = 2.67


In [25]:
def build_portfolio_returns(portfolio_results):
    """Combines individual stock returns into one equally-weighted portfolio return series."""
    returns_df = pd.DataFrame()
    
    for ticker, df in portfolio_results.items():
        returns_df[ticker] = df["Strategy_Return"]
    
    # Average across all stocks, each day = equally-weighted portfolio return that day
    returns_df["Portfolio_Return"] = returns_df.mean(axis=1)
    
    # Build portfolio's own equity curve from these combined returns
    returns_df["Portfolio_Equity"] = (1 + returns_df["Portfolio_Return"]).cumprod()
    
    return returns_df

In [26]:
portfolio_df = build_portfolio_returns(portfolio_results)
portfolio_df[["Portfolio_Return", "Portfolio_Equity"]].tail(10)

,Portfolio_Return,Portfolio_Equity
Date,,
2026-08-11,0.002951,2.129641
2026-08-12,-0.010303,2.107699
2026-08-13,0.000161,2.108038
2026-08-14,-0.003229,2.101231
2026-08-17,-0.008145,2.084116
2026-08-18,-0.007239,2.069028
2026-08-19,0.001650,2.072443
2026-08-20,0.002608,2.077848
2026-08-21,-0.001245,2.075262


In [27]:
def compare_portfolio_vs_best_single(portfolio_results, portfolio_df):
    """Compares portfolio metrics against the single best-performing stock."""
    
    # Find the single best-performing stock by final equity
    best_ticker = max(portfolio_results, key=lambda t: portfolio_results[t]["Equity_Curve"].iloc[-1])
    best_stock_df = portfolio_results[best_ticker]
    
    portfolio_metrics = calculate_metrics(portfolio_df.rename(columns={
        "Portfolio_Return": "Strategy_Return", "Portfolio_Equity": "Equity_Curve"
    }))
    best_stock_metrics = calculate_metrics(best_stock_df)
    
    comparison = pd.DataFrame({
        "Portfolio (5 stocks)": portfolio_metrics,
        f"Best Single Stock ({best_ticker})": best_stock_metrics
    })
    
    return comparison

In [28]:
comparison_table = compare_portfolio_vs_best_single(portfolio_results, portfolio_df)
comparison_table

,Portfolio (5 stocks),Best Single Stock (ITC.NS)
Total Return (%),107.34,167.16
Sharpe Ratio,0.70,0.60
Max Drawdown (%),-16.39,-28.67
Volatility (%),9.76,16.64
Win Rate (%),52.83,50.54


In [30]:
!pip install prophet

In [31]:
from prophet import Prophet

def generate_forecast(data, forecast_days=30):
    """Generates a price forecast using Prophet for the given number of future days."""
    df = data.reset_index()[["Date", "Close"]].rename(columns={"Date": "ds", "Close": "y"})
    
    model = Prophet(daily_seasonality=False)
    model.fit(df)
    
    future = model.make_future_dataframe(periods=forecast_days)
    forecast = model.predict(future)
    
    return forecast

In [32]:
forecast_result = generate_forecast(reliance_data)
forecast_result[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(10)

13:51:53 - cmdstanpy - INFO - Chain [1] start processing
13:51:54 - cmdstanpy - INFO - Chain [1] done processing


,ds,yhat,yhat_lower,yhat_upper
2898,2026-09-14,1391.857326,1309.314952,1473.064040
2899,2026-09-15,1391.615380,1304.206112,1467.392128
2900,2026-09-16,1391.856893,1314.999019,1475.880668
2901,2026-09-17,1391.052709,1312.547067,1472.973452
2902,2026-09-18,1390.706464,1316.096385,1471.011501
2903,2026-09-19,1341.110111,1263.594003,1424.305685
2904,2026-09-20,1398.779588,1312.193821,1474.987993
2905,2026-09-21,1389.292434,1306.342612,1479.013961
2906,2026-09-22,1388.562088,1306.309385,1471.986486
2907,2026-09-23,1388.303666,1310.032796,1465.268040


In [33]:
!pip install streamlit

In [2]:
import os
print(os.getcwd())

C:\Users\shahj\QuantTrack


In [2]:
import yfinance as yf
test = yf.download("RELIANCE.NS", period="5d")
print(test)


[*********************100%***********************]  1 of 1 completed

Price             Close         High          Low         Open      Volume
Ticker      RELIANCE.NS  RELIANCE.NS  RELIANCE.NS  RELIANCE.NS RELIANCE.NS
Date                                                                      
2026-09-28  1197.599976  1219.699951  1197.000000  1218.000000    13602349
2026-09-29  1182.000000  1198.300049  1181.800049  1193.800049    24191416
2026-09-30  1187.000000  1196.500000  1181.699951  1182.000000    16376789
2026-10-01  1167.699951  1183.900024  1160.800049  1180.099976    16771221


In [3]:
import sys
sys.path.append(r"C:\Users\shahj\QuantTrack")

from data_ingestion import validate_ticker, fetch_stock_data

# Test validation first
is_valid, result = validate_ticker("RELIANCE.NS")
print(f"Valid: {is_valid}, Result: {result}")

# Now test the actual fetch
data = fetch_stock_data("RELIANCE.NS")
print(data.tail() if data is not None else "Got None back")

2026-10-02 13:43:57.293 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-10-02 13:43:57.315 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


Valid: True, Result: RELIANCE.NS
Price             Close         High          Low         Open    Volume
Date                                                                    
2026-09-25  1226.000000  1227.400024  1210.500000  1210.500000  13138735
2026-09-28  1197.599976  1219.699951  1197.000000  1218.000000  13602349
2026-09-29  1182.000000  1198.300049  1181.800049  1193.800049  24191416
2026-09-30  1187.000000  1196.500000  1181.699951  1182.000000  16376789
2026-10-01  1167.699951  1183.900024  1160.800049  1180.099976  16771221
